# Module 5.3: Train Your Own GPT (Capstone)

**The goal:** take the *exact* `GPT` model you built piece-by-piece across Modules 1–4, point it at a pile of Shakespeare, and watch it learn to write — going from random gibberish to almost-Shakespeare in under three minutes on a laptop CPU.

This is the payoff.

Up to now you've built the engine part by part — embeddings (2.1), positional encoding (4.1, RoPE), attention heads (5.x), the decoder block with RMSNorm + SwiGLU (3.x), and finally the loss (5.1) and the training loop (5.2). Each module ended with *"...and later we'll put this all together."*

**Later is now.** We're going to import the assembled `GPT` from your own library and *train it for real* on a tiny dataset. No mock `nn.Sequential` like in 5.2 — this is the actual Llama-style decoder you wrote.

Think of it like a car you've spent weeks building from a kit: engine, transmission, wheels, all bolted in. This notebook is the moment you finally turn the key and drive it out of the garage.

In [ ]:
import os
import urllib.request
import torch

# Import the model YOU built across the previous modules.
from llm_workout.model import GPT

torch.manual_seed(0)

# Train on the best free hardware available. CPU is totally fine for this tiny model.
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"Training on: {device}")

## 1. The Data: a tiny mountain of Shakespeare

An LLM only ever learns one thing: *given what came before, what character (or token) comes next?* So all we need is a big blob of text. The classic teaching dataset is **TinyShakespeare** — about 1 MB of the Bard's plays, one continuous stream of characters.

We'll try, in order:
1. A local `tinyshakespeare.txt` if you already have one.
2. Otherwise download it from Karpathy's char-rnn repo.
3. If there's no network, fall back to a small embedded snippet so this notebook **always runs**.

In [ ]:
data_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
data_path = "tinyshakespeare.txt"

# A small embedded fallback so the notebook never dead-ends on a missing network.
FALLBACK_TEXT = """First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear.
""" * 40  # repeat so the fallback is big enough to actually train on

if os.path.exists(data_path):
    print(f"Found local '{data_path}'.")
    with open(data_path, 'r', encoding='utf-8') as f:
        text = f.read()
else:
    try:
        print("Downloading TinyShakespeare dataset...")
        urllib.request.urlretrieve(data_url, data_path)
        with open(data_path, 'r', encoding='utf-8') as f:
            text = f.read()
        print("Download complete.")
    except Exception as e:
        print(f"Download failed ({e}).")
        print("Falling back to a small embedded snippet so the notebook still runs.")
        print("(With this tiny corpus the model will memorize rather than truly generalize — that's OK for the demo.)")
        text = FALLBACK_TEXT

print(f"\nDataset length: {len(text):,} characters")
print("\n--- first 250 characters ---")
print(text[:250])

## 2. The simplest possible tokenizer: one character = one token

Back in **Module 2.1** we built a real **BPE** tokenizer that merges common byte-pairs into subwords. That's what GPT-4 and Llama actually use — it gives a vocabulary of tens of thousands of tokens.

Here we use something far cruder but easier to *see*: a **character-level** tokenizer. Every distinct character in the text gets its own integer id. So the vocabulary is just "the set of characters that appear in Shakespeare" — roughly 65 of them (letters, punctuation, newline).

- `stoi` ("string to int") maps a character → its id.
- `itos` ("int to string") maps an id → back to its character.
- `encode` turns a string into a list of ids; `decode` turns ids back into a string.

This is *the same idea* as BPE — text in, integers out — just with the dumbest possible vocabulary.

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"Vocabulary size: {vocab_size} unique characters")
print(f"The characters: {''.join(chars)!r}")

# Sanity check: encode then decode should round-trip.
sample = "To be, or not to be"
print(f"\n'{sample}'")
print(f"  encoded -> {encode(sample)}")
print(f"  decoded -> '{decode(encode(sample))}'")

## 3. Train/val split and `get_batch`

Two ideas to nail down here:

- **`block_size`** (a.k.a. context length): how many characters the model sees at once before predicting the next one. This is the same context window we kept referring to in the attention modules — the model can only attend to the last `block_size` tokens.
- **`batch_size`**: how many independent chunks of text we process in parallel each step. Bigger batches give a less noisy gradient estimate.

`get_batch` grabs `batch_size` random starting points and, for each, slices out `block_size` characters as the input `x` and the *same slice shifted one to the right* as the target `y`. That shift is the whole game: at every position, the target is literally "the next character."

We hold out the last 10% of the text as a **validation set** the model never trains on — so we can tell whether it's genuinely learning the language or just memorizing.

In [ ]:
batch_size = 32
block_size = 64  # context length

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"train: {len(train_data):,} chars   |   val: {len(val_data):,} chars")

def get_batch(split):
    d = train_data if split == 'train' else val_data
    # Random starting offsets, one per item in the batch.
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])          # the context
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])  # the context shifted by 1 = "next char"
    return x.to(device), y.to(device)

# Peek at one batch to make the shapes concrete.
xb, yb = get_batch('train')
print(f"\nx shape: {tuple(xb.shape)}  (batch_size, block_size)")
print(f"y shape: {tuple(yb.shape)}")
print(f"\nFor the first sequence, x[0] predicts y[0]:")
print(f"  input  : {decode(xb[0].tolist())[:40]!r}...")
print(f"  target : {decode(yb[0].tolist())[:40]!r}...   (same text, shifted one step)")

## 4. Instantiate *your* GPT

Here it is — the model you assembled, with each argument tracing straight back to a module you completed:

| Argument | What it is | Built in |
|---|---|---|
| `vocab_size` | size of the token embedding table & output head | Module 2.1 (tokenization) |
| `d_model` | width of every vector flowing through the model | Module 2.1 (embeddings) |
| `num_heads` | how many parallel attention heads per block | Module 3.1 (multi-head attention) |
| `num_layers` | how many decoder blocks stacked on top of each other | Module 4.2 (the decoder block) |
| `hidden_dim` | inner width of the SwiGLU feed-forward network | Module 4.1 (SwiGLU / RMSNorm) |
| `max_seq_len` | longest sequence RoPE is precomputed for | Module 2.2 (RoPE positional encoding) |

We deliberately pick a **small** config. The point is to *watch learning happen in a few minutes*, not to chase state of the art. (Recall the closing note of 5.2: this same loop scales unchanged up to a 400B-parameter Llama on 16,000 GPUs — only the numbers below get bigger.)

One subtle knob: `max_seq_len` sets how many positions RoPE is precomputed for. We *train* on `block_size`-long chunks, but when we **generate** we stream out hundreds of tokens one after another — and every generated token needs its own RoPE position. So `max_seq_len` must be at least as long as the longest text we plan to generate. We give it comfortable headroom (`512`).

In [ ]:
n_embd = 128       # d_model
n_head = 4
n_layer = 4
hidden_dim = n_embd * 4
max_seq_len = 512  # RoPE positions: must cover the longest text we'll generate

model = GPT(
    vocab_size=vocab_size,
    d_model=n_embd,
    num_layers=n_layer,
    num_heads=n_head,
    hidden_dim=hidden_dim,
    max_seq_len=max_seq_len,
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model instantiated with {num_params:,} parameters.")
print(f"(For scale: GPT-3 has ~175,000,000,000. This one fits in your pocket.)")

### A quick listen to the untrained model

Before training a single step, let's ask the model to generate. Its weights are random, so this should be pure noise — a baseline to measure the magic against.

We use the `generate` method you wrote — it's a generator that yields one token id at a time, driven by the **KV cache** from Module 7.1.

In [ ]:
def sample(model, n_tokens=200):
    """Generate from a single newline as the seed, return the decoded string."""
    context = torch.zeros((1, 1), dtype=torch.long, device=device)  # token id 0 as a seed
    out = [decode([context[0, 0].item()])]
    for token_id in model.generate(context, max_new_tokens=n_tokens):
        out.append(decode([token_id]))
    return ''.join(out)

print("--- BEFORE training (random weights) ---")
print(sample(model, 200))

## 5. The training loop — on the real model

This is the **exact same 5-step loop** from Module 5.2, now wrapped around your actual Transformer:

1. **Forward** — feed a batch in, get `logits` and a `loss` back (the model computes Cross-Entropy for us when we pass `targets`).
2. **Zero grad** — clear last step's gradients (they accumulate by default).
3. **Backward** — `loss.backward()` walks the graph and fills in every gradient.
4. **Step** — AdamW nudges every weight a little down the slope.

Every `eval_interval` steps we estimate train/val loss on held-out batches **and generate a sample**, so you can literally watch the text sharpen from noise into Shakespeare-ish English.

Loss intuition: with a vocab of ~65 characters, a model guessing *uniformly at random* would score about `ln(65) ≈ 4.17`. Anything well below that means it's genuinely learned which characters tend to follow which.

In [ ]:
max_iters = 800        # bump this up to train longer / get better samples
eval_interval = 200
eval_iters = 50        # how many batches to average for a stable loss estimate
learning_rate = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss(model):
    """Average the loss over several batches of train and val data (less noisy than a single batch)."""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss, _ = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

print("Starting training...\n")
for step in range(max_iters):

    # --- periodically report + show a sample so we SEE it learning ---
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss(model)
        print(f"step {step:4d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f}")
        print("   sample: " + repr(sample(model, 120)))
        print()

    # --- THE SACRED 5-STEP LOOP (from Module 5.2) ---
    xb, yb = get_batch('train')          # 1a. fresh batch
    logits, loss, _ = model(xb, yb)      # 1b. forward -> loss
    optimizer.zero_grad(set_to_none=True)  # 2. clear old gradients
    loss.backward()                      # 3. backward -> gradients
    optimizer.step()                     # 4. update weights

print("Training finished.")

Look back up at those samples. The step-0 output was random punctuation soup. By the final step it has learned word-like chunks, spaces in roughly the right places, capitalized speaker names, and line breaks — the *shape* of Shakespeare. With more iterations and a bigger model it keeps getting closer.

Let's pull a longer final sample now that it's done.

In [ ]:
print("--- AFTER training ---\n")
print(sample(model, 400))

## 6. Save a checkpoint

A trained model is just its **weights** (`state_dict`) plus the bits needed to rebuild and use it: the **config** (so we can re-instantiate the same architecture) and the **tokenizer** (`stoi`/`itos`, so generated ids map back to characters). We bundle all three into one file.

Once saved, `scripts/generate.py` can load this checkpoint and sample from your model any time — no retraining needed.

In [ ]:
checkpoint_path = "checkpoints/tiny_shakespeare.pt"
os.makedirs("checkpoints", exist_ok=True)

torch.save(
    {
        "model_state": model.state_dict(),
        "config": {
            "vocab_size": vocab_size,
            "d_model": n_embd,
            "num_layers": n_layer,
            "num_heads": n_head,
            "hidden_dim": hidden_dim,
            "max_seq_len": max_seq_len,
        },
        "stoi": stoi,
        "itos": itos,
    },
    checkpoint_path,
)
print(f"Checkpoint saved to {checkpoint_path}")
print("Sample from it later with:  python scripts/generate.py")

## You just trained a language model from scratch.

Sit with that for a second. Every component — the embeddings, RoPE, the attention math, RMSNorm, SwiGLU, the KV cache, the loss, the optimizer — you built by hand in earlier modules. Here you bolted them together and watched the thing *learn to write English* from nothing but next-character prediction. That is, in miniature, **exactly** how GPT-4 and Llama were trained. Same loop, more data, more parameters, more GPUs.

Two things we glossed over, coming next:

- **Module 5.4 — Decoding & Sampling.** Our `generate` just samples from the raw probabilities. Real systems use **temperature**, **top-k**, and **top-p (nucleus)** sampling to trade off creativity vs. coherence. Same model, very different-feeling output.
- **Module 5.5 — Evaluation.** Train/val loss is one signal, but how do we *really* measure a language model? We'll meet **perplexity** and task-based benchmarks.

And remember the cliffhanger from 5.2: this base model knows grammar and style but doesn't know how to *chat*. That's **Module 6: Fine-Tuning & Alignment**.

### 🏋️ Try it yourself

You now own the whole pipeline — go play with it. Some experiments, roughly easiest to hardest:

1. **Train longer.** Bump `max_iters` to 2000–3000 and watch the val loss keep dropping and the samples improve.
2. **Grow the model.** Try `n_layer=6`, `n_embd=192`. More capacity → better text (but slower, and watch for the train/val gap — that's overfitting).
3. **Tune the learning rate.** Try `1e-3` (faster, maybe unstable) vs `1e-4` (slower, steadier). Recall from 5.2 what too-large an `lr` does.
4. **Change the data.** Drop any `.txt` file in as `tinyshakespeare.txt` — song lyrics, code, your own writing — and train a model that talks like *that*.

The starter cell below re-runs a *quick* config so you can iterate fast. Change the knobs and run it.

In [ ]:
# Your turn: tweak these and re-run for a fast experiment.
torch.manual_seed(0)

my_n_layer = 4        # try 6
my_n_embd  = 128      # try 192
my_lr      = 3e-4     # try 1e-3 or 1e-4
my_iters   = 400      # try 1500 to really hear it sing

my_model = GPT(
    vocab_size=vocab_size,
    d_model=my_n_embd,
    num_layers=my_n_layer,
    num_heads=4,
    hidden_dim=my_n_embd * 4,
    max_seq_len=max_seq_len,
).to(device)
my_opt = torch.optim.AdamW(my_model.parameters(), lr=my_lr)
print(f"{sum(p.numel() for p in my_model.parameters()):,} parameters\n")

for step in range(my_iters):
    if step % 100 == 0:
        l = estimate_loss(my_model)
        print(f"step {step:4d} | train {l['train']:.3f} | val {l['val']:.3f}")
    xb, yb = get_batch('train')
    _, loss, _ = my_model(xb, yb)
    my_opt.zero_grad(set_to_none=True)
    loss.backward()
    my_opt.step()

print("\n--- your model's sample ---")
print(sample(my_model, 200))